# DomainTune — Notebook 1: Dataset Preparation

**Goal of this notebook:**
1. Load the full `jsdnrs/ICDAR2019-SROIE` dataset (987 receipts, train+test).
2. Inspect the actual data: missing fields, date formats, currency/total
   formats, OCR text lengths, and other edge cases — on the *complete*
   dataset, not just a handful of samples.
3. Based on what we actually find, finalize a normalization policy for the
   ground-truth `entities` (company / date / address / total). The OCR input
   text stays untouched and noisy — only the target labels get normalized.
4. Convert each receipt into an instruction-formatted training example.
5. Create train / validation / test splits and save the prepared dataset to
   disk, ready for Notebook 2 (fine-tuning).

**We do NOT fine-tune anything in this notebook.**

Run this in Google Colab (free GPU not even required for this notebook —
it's pure data work, CPU is fine).


## 1. Setup

In [1]:
!pip install -q datasets pandas


In [2]:
import re
import json
import random
from collections import Counter
from datetime import datetime

import pandas as pd
from datasets import load_dataset

random.seed(42)
pd.set_option("display.max_colwidth", 120)


## 2. Load the full dataset

We load both splits and concatenate them into one working set. We will
create our *own* train/val/test splits later (Section 7) rather than relying
on the dataset's original train/test split, so that we control the ratios
and can stratify/inspect as needed.


In [3]:
raw = load_dataset("jsdnrs/ICDAR2019-SROIE")
print(raw)


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/319M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/361 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'key', 'image_size', 'entities', 'words', 'bboxes'],
        num_rows: 626
    })
    test: Dataset({
        features: ['image', 'key', 'image_size', 'entities', 'words', 'bboxes'],
        num_rows: 361
    })
})


In [4]:
# Combine both splits into a single list of dicts for easier custom analysis.
all_rows = list(raw["train"]) + list(raw["test"])
print(f"Total receipts: {len(all_rows)}")

# Sanity check on a single row's keys/types (skip the image to keep output small)
sample = all_rows[0]
for k, v in sample.items():
    if k == "image":
        print(f"{k}: <PIL Image>")
    else:
        print(f"{k}: {type(v).__name__} -> {v if not isinstance(v, list) else v[:3]}")


Total receipts: 987
image: <PIL Image>
key: str -> X00016469612
image_size: dict -> {'width': 463, 'height': 1013}
entities: dict -> {'company': 'BOOK TA .K (TAMAN DAYA) SDN BHD', 'date': '25/12/2018', 'address': 'NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.', 'total': '9.00'}
words: list -> ['TAN WOON YANN', 'BOOK TA .K(TAMAN DAYA) SDN BND', '789417-W']
bboxes: list -> [[72, 25, 326, 64], [50, 82, 440, 121], [205, 121, 285, 139]]


## 3. Missing / empty field analysis

We check, across all 987 receipts, how often each of the four target fields
(`company`, `date`, `address`, `total`) is missing, empty, or otherwise
degenerate (e.g. whitespace-only). This directly informs how we handle
missing fields in the normalization policy (Section 6).


In [5]:
FIELDS = ["company", "date", "address", "total"]

def field_status(value):
    """Classify a raw entity field value."""
    if value is None:
        return "missing (None)"
    if isinstance(value, str) and value.strip() == "":
        return "empty string"
    return "present"

field_status_counts = {f: Counter() for f in FIELDS}
missing_examples = {f: [] for f in FIELDS}

for row in all_rows:
    ent = row["entities"]
    for f in FIELDS:
        val = ent.get(f)
        status = field_status(val)
        field_status_counts[f][status] += 1
        if status != "present" and len(missing_examples[f]) < 3:
            missing_examples[f].append(row["key"])

print("Field completeness across all", len(all_rows), "receipts:\n")
for f in FIELDS:
    print(f"  {f}:")
    for status, count in field_status_counts[f].most_common():
        pct = 100 * count / len(all_rows)
        print(f"    {status:20s}: {count:4d}  ({pct:.1f}%)")
    if missing_examples[f]:
        print(f"    example keys with issues: {missing_examples[f]}")
    print()


Field completeness across all 987 receipts:

  company:
    present             :  987  (100.0%)

  date:
    present             :  987  (100.0%)

  address:
    present             :  986  (99.9%)
    missing (None)      :    1  (0.1%)
    example keys with issues: ['X51005663280']

  total:
    present             :  986  (99.9%)
    empty string        :    1  (0.1%)
    example keys with issues: ['X51005433522']



## 4. Date format analysis

Dates in this dataset appear in many raw formats (we already saw
`25/12/2018`, `12-01-19`, `2018-03-23`, `05 MAR 2018` in the initial sample
of 5). Here we classify every date string in the full dataset against a set
of candidate formats and see:
- which formats actually occur and how often
- which date strings fail to parse under ANY of our candidate formats
  (these need manual review or a `date_parse_failed` flag)


In [6]:
# Candidate date formats. Extended after running against the full 987-row
# dataset in Colab/Kaggle and inspecting the 33 strings that didn't match
# our original list — every one of these turned out to be a legitimate,
# recognizable format, just one we hadn't enumerated yet:
#   "28 MAR 18"     -> %d %b %y
#   "24-MAR-2018"   -> %d-%b-%Y
#   "11.02.18"      -> %d.%m.%y
#   "02/JAN/2017"   -> %d/%b/%Y
#   "(06/12/2016)"  -> %d/%m/%Y, once we strip stray parens
#   "OCT 3, 2016"   -> %b %d, %Y
DATE_FORMATS = [
    "%d/%m/%Y", "%d-%m-%Y", "%d/%m/%y", "%d-%m-%y",
    "%Y-%m-%d", "%Y/%m/%d",
    "%d %b %Y", "%d %B %Y",
    "%d %b %y", "%d %B %y",
    "%d-%b-%Y", "%d-%B-%Y",
    "%d/%b/%Y", "%d/%B/%Y",
    "%d.%m.%Y", "%d.%m.%y",
    "%b %d, %Y", "%B %d, %Y",
    "%m/%d/%Y", "%m-%d-%Y",
    "%Y%m%d", "%d%m%Y",
]

def try_parse_date(raw_date):
    """Try each candidate format; return (parsed_date, matched_format) or (None, None).

    Strips a few harmless wrapping/formatting artifacts first (stray
    parentheses, as in "(06/12/2016)") before attempting to match.
    """
    if not raw_date or not raw_date.strip():
        return None, None
    cleaned = raw_date.strip().strip("()").strip()
    for fmt in DATE_FORMATS:
        try:
            dt = datetime.strptime(cleaned, fmt)
            return dt, fmt
        except ValueError:
            continue
    return None, None

format_match_counts = Counter()
unparsed_dates = []

for row in all_rows:
    raw_date = row["entities"].get("date")
    if not raw_date:
        continue
    dt, fmt = try_parse_date(raw_date)
    if fmt:
        format_match_counts[fmt] += 1
    else:
        unparsed_dates.append((row["key"], raw_date))

print("Date format match counts (across all non-empty date fields):\n")
for fmt, count in format_match_counts.most_common():
    print(f"  {fmt:15s}: {count:4d}")

print(f"\nUnparsed dates: {len(unparsed_dates)} / "
      f"{sum(format_match_counts.values()) + len(unparsed_dates)}")
print("Sample of unparsed date strings (up to 20):")
for key, d in unparsed_dates[:20]:
    print(f"  {key}: {d!r}")


Date format match counts (across all non-empty date fields):

  %d/%m/%Y       :  536
  %d-%m-%y       :  114
  %d-%m-%Y       :  110
  %d/%m/%y       :   97
  %d %b %Y       :   84
  %d/%b/%Y       :   10
  %d %b %y       :    9
  %d-%b-%Y       :    9
  %Y-%m-%d       :    6
  %m/%d/%Y       :    3
  %Y/%m/%d       :    3
  %Y%m%d         :    2
  %d.%m.%y       :    2
  %d%m%Y         :    1
  %b %d, %Y      :    1

Unparsed dates: 0 / 987
Sample of unparsed date strings (up to 20):


## 5. Total / currency format analysis

We already spotted `9.00`, `60.30`, `$8.20`, and `RM 3.90`-style variants in
the initial sample. Here we classify every `total` value in the full dataset
by its currency-notation pattern so we know exactly which symbols/formats
the normalization function needs to strip, and how often the field is
non-numeric or otherwise malformed.


In [7]:
# Patterns we expect to see. Extended after the full-dataset run turned up
# 3 negative totals (-1.73, -6.42, -5.09) — real credit-note/refund
# receipts, not malformed data, so we keep the sign as a valid pattern
# rather than discarding it.
CURRENCY_PATTERNS = {
    "plain_decimal":        r"^-?\d+\.\d{1,2}$",            # 9.00, -1.73
    "plain_integer":        r"^-?\d+$",                        # 9, -6
    "dollar_prefix":        r"^\$\s*-?\d+\.\d{1,2}$",       # $8.20
    "rm_prefix":            r"^RM\s*-?\d+\.\d{1,2}$",        # RM 3.90
    "comma_thousands":      r"^-?\d{1,3}(,\d{3})+\.\d{1,2}$", # 1,234.56
}

def classify_total(raw_total):
    if raw_total is None or raw_total.strip() == "":
        return "empty_or_missing"
    stripped = raw_total.strip()
    for name, pattern in CURRENCY_PATTERNS.items():
        if re.match(pattern, stripped):
            return name
    return "unrecognized"

total_pattern_counts = Counter()
unrecognized_totals = []

for row in all_rows:
    raw_total = row["entities"].get("total")
    category = classify_total(raw_total)
    total_pattern_counts[category] += 1
    if category == "unrecognized":
        unrecognized_totals.append((row["key"], raw_total))

print("Total-field format distribution:\n")
for cat, count in total_pattern_counts.most_common():
    pct = 100 * count / len(all_rows)
    print(f"  {cat:20s}: {count:4d}  ({pct:.1f}%)")

print(f"\nUnrecognized total strings (up to 30 shown):")
for key, t in unrecognized_totals[:30]:
    print(f"  {key}: {t!r}")


Total-field format distribution:

  plain_decimal       :  836  (84.7%)
  rm_prefix           :   99  (10.0%)
  dollar_prefix       :   50  (5.1%)
  empty_or_missing    :    1  (0.1%)
  comma_thousands     :    1  (0.1%)

Unrecognized total strings (up to 30 shown):


## 6. OCR text length and structure analysis

We inspect the `words` list (our OCR input) across all receipts: how many
lines per receipt, how long the concatenated text is, and whether any
receipts are unusually short/long (possible bad scans or parsing errors
worth excluding or flagging).


In [8]:
word_counts = []
char_counts = []

for row in all_rows:
    words = row["words"]
    joined = "\n".join(words)
    word_counts.append(len(words))
    char_counts.append(len(joined))

word_counts_s = pd.Series(word_counts)
char_counts_s = pd.Series(char_counts)

print("OCR lines per receipt (words list length):")
print(word_counts_s.describe())
print()
print("Newline-joined character length per receipt:")
print(char_counts_s.describe())

# Flag outliers: very short receipts may indicate a bad scan / OCR failure.
SHORT_LINE_THRESHOLD = 10
short_receipts = [
    row["key"] for row in all_rows if len(row["words"]) < SHORT_LINE_THRESHOLD
]
print(f"\nReceipts with fewer than {SHORT_LINE_THRESHOLD} OCR lines: "
      f"{len(short_receipts)}")
print(short_receipts[:20])


OCR lines per receipt (words list length):
count    987.000000
mean      53.710233
std       17.656031
min       18.000000
25%       42.000000
50%       50.000000
75%       63.000000
max      153.000000
dtype: float64

Newline-joined character length per receipt:
count     987.000000
mean      626.457953
std       174.229457
min       169.000000
25%       501.000000
50%       592.000000
75%       727.000000
max      1323.000000
dtype: float64

Receipts with fewer than 10 OCR lines: 0
[]


In [9]:
# Look at a few of the shortest receipts directly, to decide whether they're
# genuinely unusable or just naturally short receipts.
shortest = sorted(all_rows, key=lambda r: len(r["words"]))[:3]
for row in shortest:
    print(f"--- {row['key']} ({len(row['words'])} lines) ---")
    print("\n".join(row["words"]))
    print("entities:", row["entities"])
    print()


--- X51005442341 (18 lines) ---
RESTAURANT SIN DU
K3-113
80300 JOHOR BAHRU
JOHOR
H/P: 019-7521215
016-7867868
09/03/2018 21:28
0001
000000#7259 CASHIER01
DPT.05
RM
149.00
DPT.04
RM
21.00
CASH
RM
170.00
entities: {'company': 'RESTAURANT SIN DU', 'date': '09/03/2018', 'address': 'K3-113, JL IBRAHIM SULTAN 80300 JOHOR BAHRU JOHOR', 'total': '170.00'}

--- X51008123446 (22 lines) ---
DION REALTIES SDN BHD (CO. NO:20154-T)
(GST REGISTRATION NO : 000650247680)
MENARA DION #02-03
27
50250 KUALA LUMPUR.
TEL : + 6 03 2026 6386
FAX : +6 03 2026 6387
TAX INVOICE
TAX INVOICE NO. 3521/0602/00602
30/05/18 11:01
010100 PAY PARKING TICKET
5.00 RM
30/05/18 10:45 - 30/05/18 11:01
LENGTH OF STAY: 0 DY. 0 HR. 16 MIN.
02992887002011018150387040??
AMOUNT INCL. GST
5.00 RM
ACCEPTAD TOTAL
5.00 RM
GST 6%
0.28 RM
THANK YOU
entities: {'company': 'DION REALTIES SDN BHD', 'date': '30/05/18', 'address': 'MENARA DION #02-03, LEVEL 2, 27, JALAN SULTAN ISMAIL, 50250 KUALA LUMPUR.', 'total': '5.00'}

--- X51008123450 (

## 7. Duplicate / near-duplicate receipt check

The initial 5-row sample already showed what look like repeat visits to the
same merchant (e.g. multiple "UNIHAKKA INTERNATIONAL SDN BHD" receipts with
different dates). That's expected and fine for training diversity, but exact
duplicate receipts (same `key` twice, or identical `words`) would be a data
leakage risk between splits, so we check for those specifically.

**Finding from the full-dataset run:** 0 duplicate `key`s, but 2 pairs of
byte-identical OCR content under *different* keys — and a naive shuffle-then-
split let one of those pairs land in different splits (train/val), which is
a real leakage risk. We fix this in Section 10 by grouping duplicate-content
receipts together before splitting, so a pair always lands in the same
split rather than being dropped or allowed to leak.


In [10]:
keys = [row["key"] for row in all_rows]
key_counts = Counter(keys)
duplicate_keys = {k: c for k, c in key_counts.items() if c > 1}
print(f"Duplicate receipt keys: {len(duplicate_keys)}")
print(duplicate_keys)

# Exact duplicate OCR content (different key, identical words) — grouped so
# that Section 10 can keep each group together in a single split rather than
# letting duplicates leak across train/val/test.
content_to_keys = {}
for row in all_rows:
    content = "\n".join(row["words"])
    content_to_keys.setdefault(content, []).append(row["key"])

duplicate_content_groups = {c: ks for c, ks in content_to_keys.items() if len(ks) > 1}
print(f"\nReceipts with exactly duplicated OCR text content: "
      f"{sum(len(ks) for ks in duplicate_content_groups.values())} "
      f"across {len(duplicate_content_groups)} group(s)")
for content, ks in duplicate_content_groups.items():
    print(f"  group: {ks}")


Duplicate receipt keys: 0
{}

Receipts with exactly duplicated OCR text content: 4 across 2 group(s)
  group: ['X51006328919', 'X51006329395']
  group: ['X51007103641', 'X51007103643']


## 8. Finalize the normalization policy

**Finalized based on the actual full-dataset run** (987 receipts, executed
in Colab/Kaggle — see accompanying results): 33 dates initially failed to
parse (all legitimate formats, now added to `DATE_FORMATS` in Section 4);
3 totals were negative, real credit-note/refund values (now explicitly
supported, sign preserved); 1 address and 1 total were genuinely missing.

Guiding principle (agreed): **keep the OCR input noisy; only normalize the
ground-truth target.**

- **company**: trim outer whitespace, collapse internal multi-spaces. No
  case changes.
- **date**: parse into canonical `YYYY-MM-DD` using the extended format
  list from Section 4 (now covers month-abbreviation, dot-separated,
  parenthesis-wrapped, and US comma-style dates found in the real data). If
  still unparseable, keep the raw string AND set `date_parse_failed = True`
  so we can filter or inspect these later, rather than silently guessing.
- **total**: strip currency symbols/text and thousands separators, keep the
  numeric value as a plain decimal string, preserving a leading `-` for
  genuine refund/credit-note receipts (e.g. `"-1.73"`). If empty/missing or
  non-numeric after stripping, set to `None` explicitly.
- **address**: trim whitespace, collapse multi-spaces/commas. Leave content
  otherwise untouched.
- **Missing fields**: represented as `None` in the target JSON (not `""`),
  so "genuinely absent" is distinguishable from "extracted as empty."


In [11]:
def normalize_company(raw):
    if raw is None or raw.strip() == "":
        return None
    return re.sub(r"\s+", " ", raw.strip())


def normalize_address(raw):
    if raw is None or raw.strip() == "":
        return None
    cleaned = re.sub(r"\s+", " ", raw.strip())
    cleaned = re.sub(r"\s*,\s*", ", ", cleaned)
    return cleaned


def normalize_date(raw):
    """Return (normalized_date_or_None, parse_failed: bool)."""
    if raw is None or raw.strip() == "":
        return None, False  # genuinely missing, not a parse failure
    dt, fmt = try_parse_date(raw)
    if dt is not None:
        return dt.strftime("%Y-%m-%d"), False
    return raw.strip(), True  # keep raw string, flag as failed to parse


def normalize_total(raw):
    """Strip currency notation, keep sign (refund/credit-note receipts
    legitimately have negative totals), keep two decimal places."""
    if raw is None or raw.strip() == "":
        return None
    cleaned = raw.strip()
    cleaned = re.sub(r"(?i)^rm\s*", "", cleaned)   # leading "RM"
    cleaned = cleaned.replace("$", "")
    cleaned = cleaned.replace(",", "")
    cleaned = cleaned.strip()
    if re.match(r"^-?\d+(\.\d{1,2})?$", cleaned):
        value = float(cleaned)
        return f"{value:.2f}"
    return None


def normalize_entities(raw_entities):
    company = normalize_company(raw_entities.get("company"))
    address = normalize_address(raw_entities.get("address"))
    date, date_parse_failed = normalize_date(raw_entities.get("date"))
    total = normalize_total(raw_entities.get("total"))
    return {
        "company": company,
        "date": date,
        "address": address,
        "total": total,
        "_date_parse_failed": date_parse_failed,
    }


# Quick spot check against a handful of rows, including one of the
# previously-negative totals and a previously-unparsed date, to confirm the
# fixes work as intended.
spot_check_keys = {"X00016469612", "X51006556824", "X51005444037"}
for row in all_rows:
    if row["key"] in spot_check_keys:
        print(row["key"])
        print("  raw:  ", row["entities"])
        print("  norm: ", normalize_entities(row["entities"]))
        print()


X00016469612
  raw:   {'company': 'BOOK TA .K (TAMAN DAYA) SDN BHD', 'date': '25/12/2018', 'address': 'NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.', 'total': '9.00'}
  norm:  {'company': 'BOOK TA .K (TAMAN DAYA) SDN BHD', 'date': '2018-12-25', 'address': 'NO.53 55, 57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.', 'total': '9.00', '_date_parse_failed': False}

X51005444037
  raw:   {'company': "NANDO'S CHICKENLAND MALAYSIA SDN BHD", 'date': '28 MAR 18', 'address': 'UNIT G-13, GROUND FLOOR, NO.1 JLN KIARA, MONT KIARA 50480 KUALA LUMPUR', 'total': '129.30'}
  norm:  {'company': "NANDO'S CHICKENLAND MALAYSIA SDN BHD", 'date': '2018-03-28', 'address': 'UNIT G-13, GROUND FLOOR, NO.1 JLN KIARA, MONT KIARA 50480 KUALA LUMPUR', 'total': '129.30', '_date_parse_failed': False}

X51006556824
  raw:   {'company': 'GARDENIA BAKERIES (KL) SDN BHD', 'date': '29/09/2017', 'address': 'LOT 3, JALAN PELABUR 23/1, 40300 SHAH ALAM, SELANGOR.', 'total': '-1.73'}
  no

In [12]:
# Run normalization across the full dataset and quantify the impact:
# how many examples have a parse-failed date, a None total, etc. This tells
# us whether our policy needs another look before we build training examples.
normalized_all = [normalize_entities(row["entities"]) for row in all_rows]

n_date_failed = sum(1 for e in normalized_all if e["_date_parse_failed"])
n_total_none = sum(1 for e in normalized_all if e["total"] is None)
n_company_none = sum(1 for e in normalized_all if e["company"] is None)
n_address_none = sum(1 for e in normalized_all if e["address"] is None)

print(f"Total receipts: {len(normalized_all)}")
print(f"  date parse failed : {n_date_failed}")
print(f"  total -> None     : {n_total_none}")
print(f"  company -> None   : {n_company_none}")
print(f"  address -> None   : {n_address_none}")

# TODO: decide, based on these counts, whether to:
#   (a) exclude examples with a None total from training (a null target
#       teaches the model nothing useful for that field), or
#   (b) keep them and let the model learn to output null when appropriate.
# Revisit after seeing the real numbers.


Total receipts: 987
  date parse failed : 0
  total -> None     : 1
  company -> None   : 0
  address -> None   : 1


## 9. Build instruction-formatted training examples

Input = newline-joined OCR `words` (noisy, untouched).
Output = normalized `entities` JSON (the internal `_date_parse_failed` flag
is dropped from the final target — it's a QA aid, not something the model
should predict).


In [13]:
INSTRUCTION = (
    "Extract the company name, date, address, and total amount from this "
    "receipt OCR text. Respond with a JSON object with keys: "
    "company, date, address, total."
)

def build_example(row):
    ocr_text = "\n".join(row["words"])
    normalized = normalize_entities(row["entities"])
    target = {
        "company": normalized["company"],
        "date": normalized["date"],
        "address": normalized["address"],
        "total": normalized["total"],
    }
    return {
        "key": row["key"],
        "instruction": INSTRUCTION,
        "input": ocr_text,
        "output": json.dumps(target, ensure_ascii=False),
        "_date_parse_failed": normalized["_date_parse_failed"],
    }

examples = [build_example(row) for row in all_rows]
print(f"Built {len(examples)} examples.")
print()
print("--- Example ---")
ex = examples[0]
print("INSTRUCTION:", ex["instruction"])
print("INPUT:\n", ex["input"][:300], "...")
print("OUTPUT:", ex["output"])


Built 987 examples.

--- Example ---
INSTRUCTION: Extract the company name, date, address, and total amount from this receipt OCR text. Respond with a JSON object with keys: company, date, address, total.
INPUT:
 TAN WOON YANN
BOOK TA .K(TAMAN DAYA) SDN BND
789417-W
NO.53 55
TAMAN DAYA
81100 JOHOR BAHRU
JOHOR.
DOCUMENT NO : TD01167104
DATE:
25/12/2018 8:13:39 PM
CASHIER:
MANIS
MEMBER:
CASH BILL
CODE/DESC
PRICE
DISC
AMOUNT
QTY
RM
RM
9556939040116
KF MODELLING CLAY KIDDY FISH
1 PC
*
9.000
0.00
9.00
TOTAL:
ROUR ...
OUTPUT: {"company": "BOOK TA .K (TAMAN DAYA) SDN BHD", "date": "2018-12-25", "address": "NO.53 55, 57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.", "total": "9.00"}


## 10. Train / validation / test splits

We create our own splits (rather than using the dataset's original
train/test division) so we control the ratios directly. Default: 80% train,
10% validation, 10% test, shuffled with a fixed seed for reproducibility.

**Duplicate-content groups (Section 7) are kept together as a single unit**
during splitting — we shuffle at the group level, not the individual-example
level, so a pair of byte-identical receipts always lands in the same split.
This fixes the leak we found in the earlier run (one duplicate pair split
across train/val) without dropping any examples.


In [14]:
# Build groups: each duplicate-content set is one group; every other example
# is its own singleton group. We shuffle and split at the GROUP level so
# duplicate-content receipts can never end up in different splits.
key_to_example = {ex["key"]: ex for ex in examples}
grouped_keys = set()
groups = []

for content, ks in duplicate_content_groups.items():
    groups.append(ks)
    grouped_keys.update(ks)

for ex in examples:
    if ex["key"] not in grouped_keys:
        groups.append([ex["key"]])

random.shuffle(groups)

n_total_examples = len(examples)
n_train_target = int(n_total_examples * 0.8)
n_val_target = int(n_total_examples * 0.1)

train_examples, val_examples, test_examples = [], [], []
running_count = 0

for group in groups:
    group_examples = [key_to_example[k] for k in group]
    if running_count < n_train_target:
        train_examples.extend(group_examples)
    elif running_count < n_train_target + n_val_target:
        val_examples.extend(group_examples)
    else:
        test_examples.extend(group_examples)
    running_count += len(group_examples)

print(f"Train: {len(train_examples)}")
print(f"Val:   {len(val_examples)}")
print(f"Test:  {len(test_examples)}")

# Verify no leakage: every duplicate-content group should be entirely
# contained within exactly one split.
split_of_key = {}
for split_name, split_rows in [("train", train_examples), ("val", val_examples), ("test", test_examples)]:
    for ex in split_rows:
        split_of_key[ex["key"]] = split_name

leak_found = False
for content, ks in duplicate_content_groups.items():
    splits_used = {split_of_key[k] for k in ks}
    if len(splits_used) > 1:
        leak_found = True
        print(f"LEAK: group {ks} spans splits {splits_used}")

print("No duplicate-content leakage across splits." if not leak_found else "LEAKAGE DETECTED — see above.")


Train: 789
Val:   98
Test:  100
No duplicate-content leakage across splits.


In [15]:
import os

os.makedirs("data/prepared", exist_ok=True)

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            # Drop the internal QA flag before saving the final training file.
            clean_row = {k: v for k, v in row.items() if not k.startswith("_")}
            f.write(json.dumps(clean_row, ensure_ascii=False) + "\n")

save_jsonl("data/prepared/train.jsonl", train_examples)
save_jsonl("data/prepared/val.jsonl", val_examples)
save_jsonl("data/prepared/test.jsonl", test_examples)

print("Saved train/val/test JSONL files to data/prepared/")


Saved train/val/test JSONL files to data/prepared/


## 11. Export as a ZIP (for Kaggle)

Kaggle notebooks don't give you a simple file browser to grab individual
files the way Colab does — the easiest way to get the prepared dataset out
is to zip it and download it from the notebook's output panel (or from
`/kaggle/working/` in the file browser on the right).


In [16]:
import shutil

zip_path = shutil.make_archive("prepared_dataset", "zip", "data/prepared")
print(f"Created ZIP: {zip_path}")

# On Kaggle, anything written to /kaggle/working/ is automatically picked up
# as a notebook output you can download from the "Output" tab / file browser
# on the right, regardless of the notebook's current working directory — so
# no extra copy step is needed there. If you're running this in Colab
# instead, use the Files panel on the left to download `prepared_dataset.zip`
# directly, or mount Drive and copy it there.


Created ZIP: /kaggle/working/prepared_dataset.zip


## 12. Next steps

- Review the outputs of Sections 3–7 together and adjust the normalization
  policy / edge-case handling in Section 8 if the real distributions turn
  up anything unexpected (e.g. a date format we didn't anticipate, or a
  higher-than-expected rate of missing totals).
- Once the policy is finalized and the prepared JSONL files look correct,
  move to **Notebook 2: fine-tuning** — loading Qwen 2.5 0.5B Instruct,
  configuring 4-bit quantization + LoRA, running a sanity check, then the
  actual SFT run.
